# 🚀 Resume Dataset Analysis

**Author:** Asset Nakupov  
**Date:** 2025-08-01  

---

## Overview

As someone who has made the transition from law to tech, I am curious about how common this career shift really is. In order to find this out, I have used a structured dataset of 54,0000 resumes to run a data analysis project. 

This notebook explores a structured resume dataset to uncover:

1. **Educational background distributions** across tech roles  
2. **Career pathways** of law graduates in IT  
3. **Degree profiles** for junior vs. senior developers  
4. **Specialty fields** within legal and tech professions

I combine SQL queries via `pandasql` for fast grouping and joins,  
then use `pandas` + `seaborn` for data wrangling and visualization.  
Data cleaning is handled by a custom `resume_data_cleaner` module.

This analysis asks: How do education paths shape careers in tech and law? We'll use SQL and Python to answer clear questions about degrees and roles.

In [ ]:
import kagglehub
import pandas
import pandasql
import os
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/kaggle/input/helper') 
sns.set_theme(style="darkgrid")

### Step 1: Download & Load the Data

We pull in the structured resume dataset from Kaggle and load each CSV into a pandas DataFrame.  

In [ ]:
resume_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/01_people.csv")
abilities_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/02_abilities.csv")
education_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/03_education.csv")
experience_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/04_experience.csv")
person_skills_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/05_person_skills.csv")
skills_table = pandas.read_csv("/kaggle/input/resume-dataset-structured/06_skills.csv")

### STEP 2: DATA CLEANING

We import the data cleaning module, and run each DataFrame through it to fill missing values, drop invalid rows, standardize column names and drop duplicate rows.

In [ ]:
import resume_data_cleaner as rdc
print("✅ Imported resume_data_cleaner")

resume_table_clean = rdc.resume_data_cleaner(resume_table)
print("✅ Cleaned resume_table")

abilities_table_clean = rdc.abilities_data_cleaner(abilities_table)
print("✅ Cleaned abilities_table")

education_table_clean = rdc.education_data_cleaner(education_table)
print("✅ Cleaned education_table")

experience_table_clean = rdc.experience_data_cleaner(experience_table)
print("✅ Cleaned experience_table")

person_skills_table_clean = rdc.person_skills_data_cleaner(person_skills_table)
print("✅ Cleaned person_skills_table")

skills_table_clean = rdc.skills_data_cleaner(skills_table)
print("✅ Cleaned skills_table")

print("🎉 All cleaning steps completed successfully!")


### STEP 3: SQL and Plotting

We apply custom written SQL queries to the DataFrames, filter out unknown or incorrect data, and plot the result into `Seaborn` visualizations.

# 1. Developer Degree Distribution

Question: What degrees do software developers actually hold?

In [ ]:
# Apply the SQL query to count the number of people in each degree category
with open('/kaggle/input/sql-files/01_dev_degree_distribution.sql') as f:
    education_query = f.read()

education_query_result = pandasql.sqldf(education_query, locals())

# Remove rows where the degree is 'Unknown'
# ~ is the "not" operator.
filtered_df = education_query_result[
    ~education_query_result['degree'].isin(['Unknown'])
]

# Count and sort the number of people in each degree category
datapoint_1 = filtered_df['degree'].value_counts().sort_values(ascending=False)

# What I have so far is pandas.Series
# A series isn't a table, it's more like a list of tuples with the label and the value 
# A DataFrame is a table with columns
# reset_index() converts the Series into a table (dataframe)
# now we have columns! they need names
df1 = datapoint_1.reset_index().rename(columns={'index': 'degree', 0: 'count'})

# Create a ranked bar chart with Seaborn
plt.figure(figsize=(10, 5))  # Set figure size
sns1 = sns.barplot(
    data=df1,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

# Add labels and title
sns1.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Ranked Degree Distribution for Developers"
)

# Display value labels on bars
# In plt and Seaborn, a patch is any filled in shape
# So, .patches is the list over which we can iterate to do something to every filled in shape (like a bar in a bar graph) 
total1 = df1['count'].sum()
for element in sns1.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total1 * 100
    sns1.text(
        element.get_width() + total1*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df1['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### Just below half of all developers hold Computer Science (and related) degrees. Business and Art majors follow, combining for just over 17% of all developers. Law graduates (such as myself) working in IT fields totaled to only 450 professionals out of 50,000 resumes in the dataset.

# 2. IT Degrees by Role

Question: Do front-end, back-end, full-stack, and data scientists come from different degree backgrounds? We’ll break down degrees by each role.

In [ ]:
# SQL to group compute
with open('/kaggle/input/sql-files/02a_it_roles_degree_by_field.sql') as f:
    it_fields_query = f.read()

it_fields_query_result = pandasql.sqldf(it_fields_query, locals())
filtered_it_fields = it_fields_query_result[
    ~it_fields_query_result['field'].isin(['Unknown']) & ~it_fields_query_result['degree'].isin(['Unknown'])
]

## 2A. Degrees of Front-End Developers

In [ ]:
it_fields_a = filtered_it_fields[filtered_it_fields['field'] == 'Front-end']
datapoint_6a = it_fields_a['degree'].value_counts().sort_values(ascending=False)
df6a = datapoint_6a.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns6a = sns.barplot(
    data=df6a,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns6a.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Front-end Developers"
)

total6a = df6a['count'].sum()
for element in sns6a.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total6a * 100
    sns6a.text(
        element.get_width() + total6a*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df6a['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout()

### Front-end developers have the smallest percentage of computer sceince graduates among the fields of IT industry explored in this Datapoint 2 at 39%. Art majors comprise almost 13% of all front-end specialists.

## 2B. Begrees of Back-End Developers

In [ ]:
it_fields_b = filtered_it_fields[filtered_it_fields['field'] == 'Back-end']
datapoint_6b = it_fields_b['degree'].value_counts().sort_values(ascending=False)
df6b = datapoint_6b.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns6b = sns.barplot(
    data=df6b,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns6b.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Back-end Developers"
)

total6b = df6b['count'].sum()
for element in sns6b.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total6b * 100
    sns6b.text(
        element.get_width() + total6b*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df6b['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout()

### Back-end developers are significantly more likely to hold a computer science degree, at 59.2%, which is a 20% increase as compared to front-end developers. Art majors fall of dramatically, as communications and science are the second and thirt most common major for back-end developers.

## 2C. Degrees of Full-Stack Developers

In [ ]:
it_fields_c = filtered_it_fields[filtered_it_fields['field'] == 'Full-stack']
datapoint_6c = it_fields_c['degree'].value_counts().sort_values(ascending=False)
df6c = datapoint_6c.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns6c = sns.barplot(
    data=df6c,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns6c.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Full-stack Developers"
)

total6c = df6c['count'].sum()
for element in sns6c.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total6c * 100
    sns6c.text(
        element.get_width() + total6c*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df6c['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### Full-stack developers follow the trend set by back-end developers - just over 57% of all full-stack specialists hold computer science and related degrees. Please note that there are a lot fewer professionals in the dataset who describe their job title as "full-stack" (or alternative spellings), which might potentially make this number less reliable.

## 2D. Degrees of Data Scientists

In [ ]:
it_fields_d = filtered_it_fields[filtered_it_fields['field'] == 'Data Science']
datapoint_6d = it_fields_a['degree'].value_counts().sort_values(ascending=False)
df6d = datapoint_6d.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns6d = sns.barplot(
    data=df6d,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns6d.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Data Scientists"
)

total6d = df6d['count'].sum()
for element in sns6d.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total6d * 100
    sns6d.text(
        element.get_width() + total6d*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df6d['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### IT workers working with data (data scientists, data engineers, data analysts) follow the trend set by front-end developers: 39% have computer science and related degrees, with art being the second most common major.

## 2E. Degrees of Senior Developers

In [ ]:
with open('/kaggle/input/sql-files/02b_it_degree_by_seniority.sql') as f:
    it_fields_sr_jr_query = f.read()

it_fields_sr_jr_query_result = pandasql.sqldf(it_fields_sr_jr_query, locals())
filtered_it_fields_sr_jr = it_fields_sr_jr_query_result[
    ~it_fields_sr_jr_query_result['level'].isin(['Unknown']) & ~it_fields_sr_jr_query_result['degree'].isin(['Unknown'])
]

it_fields_7 = filtered_it_fields_sr_jr[filtered_it_fields_sr_jr['level'] == 'Senior']
datapoint_7 = it_fields_7['degree'].value_counts().sort_values(ascending=False)
df7 = datapoint_7.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns7 = sns.barplot(
    data=df7,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns7.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Senior Developers"
)

total7 = df7['count'].sum()
for element in sns7.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total7 * 100
    sns7.text(
        element.get_width() + total7*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df7['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout()

### This dataset groups specialists by seniority, as compared to field of IT (in previous datapoints). 47% of senior developers hold IT related degrees, with business being the second most common major.

## 2F. Degrees of Junior Developers

In [ ]:
it_fields_8 = filtered_it_fields_sr_jr[filtered_it_fields_sr_jr['level'] == 'Junior']
datapoint_8 = it_fields_8['degree'].value_counts().sort_values(ascending=False)
df8 = datapoint_8.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns8 = sns.barplot(
    data=df8,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns8.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Junior Developers"
)

total8 = df8['count'].sum()
for element in sns8.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total8 * 100
    sns8.text(
        element.get_width() + total8*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df8['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout()

### Almost 43% of junior developers hold IT related degrees, a 4% decrease as compared to senior developers.

# 3. Law Graduates in IT Fields

Question: Which IT fields do law grads enter? We’ll chart the IT roles held by people with law degrees.

In [ ]:
with open('/kaggle/input/sql-files/03_law_grads_it_fields.sql') as f:
    only_lawyers_query = f.read()

only_lawyers_query_result = pandasql.sqldf(only_lawyers_query, locals())
datapoint_2 = only_lawyers_query_result['field'].value_counts().sort_values(ascending=False)
df2 = datapoint_2.reset_index().rename(columns={'index': 'field', 0: 'count'})

plt.figure(figsize=(13, 7))
sns2 = sns.barplot(
    data=df2,
    y='count',
    x='field',
    orient='v',
    edgecolor="black"
)

sns2.set(
    xlabel="Number of people",
    ylabel="IT Field",
    title="Fields of IT that Law Graduates Work In"
)

total2 = df2['count'].sum()
for element in sns2.patches:
    count = element.get_height()
    percentage = count/total2 * 100
    sns2.text(
        element.get_x(),
        count + df2['count'].max()*0.01,
        f"{count} ({percentage:.1f}%)",
        va="bottom", # vertical align
    )

plt.xticks(rotation=45, ha="right") # Rotate the x-axis labels
plt.ylim(0, df2['count'].max() * 1.1)
plt.tight_layout()

### Surprisingly, law graduates are most likely to work in front-end and web development, with more than half of all people with law and related degrees who work in IT being front-end specialists.

# 4. Degrees of Legal Workers

This question is a reversal of question 1. We learned what degrees developers hold, but what degrees do legal professionals hold? We’ll rank degrees among those with legal job titles.

In [ ]:
with open('/kaggle/input/sql-files/04_law_grads_degree_breakdown.sql') as f:
    lawyers_degrees_query = f.read()

lawyers_degrees_query_result = pandasql.sqldf(lawyers_degrees_query, locals())
filtered_lawyers_degrees = lawyers_degrees_query_result[
    ~lawyers_degrees_query_result['degree'].isin(['Unknown'])
]
datapoint_3 = filtered_lawyers_degrees['degree'].value_counts().sort_values(ascending=False)
df3 = datapoint_3.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns3 = sns.barplot(
    data=df3,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns3.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Ranked Degree Distribution for Legal Workers"
)

total3 = df3['count'].sum()
for element in sns3.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total3 * 100
    sns3.text(
        element.get_width() + total3*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df3['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### Surprisingly, people in the legal field are a lot less likely to have a law (and related, e.g. criminology, JD, legal studies, etc.) degree, with only 26.1% of legal workers holding a law degree.

# 5. Legal Worker Job Breakdown

Question: What specific roles exist in the legal field? Count each position (e.g., Lawyer, Paralegal, Judge).

In [ ]:
with open('/kaggle/input/sql-files/05_law_grads_positions.sql') as f:
    lawyers_jobs_query = f.read()

lawyers_jobs_query_result = pandasql.sqldf(lawyers_jobs_query, locals())
filtered_lawyers_jobs = lawyers_jobs_query_result[
    ~lawyers_jobs_query_result['position'].isin(['Unknown']) & ~lawyers_degrees_query_result['degree'].isin(['Unknown'])
]
datapoint_4 = filtered_lawyers_jobs['position'].value_counts().sort_values(ascending=False)
df4 = datapoint_4.reset_index().rename(columns={"index": 'position', 0: 'count'})

plt.figure(figsize=(10, 5))
sns4 = sns.barplot(
    data=df4,
    y='position',
    x='count',
    orient='h',
    edgecolor="black"
)

sns4.set(
    xlabel="Number of People",
    ylabel="Position",
    title="Positions of Legal Workers"
)

total4 = df4['count'].sum()
for element in sns4.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total4 * 100
    sns4.text(
        element.get_width() + total4*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df4['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### The above chart breaks down the positions that the legal workers in the dataset hold. Most legal specialists hold relatively junior or support positions: paralegals, assistants and secretaries comprise over 58% of all legal sphere workers.

# 6. Degrees of Lawyers (Excluding Paralegals, Secretaries and Assistants) 

Question: Among core lawyers, what degrees stand out? Filter out support roles and rank degrees.

In [ ]:
only_senior_lawyers = lawyers_jobs_query_result[
    ~lawyers_jobs_query_result['position'].isin(['Unknown', 'Assistant', 'Secretary', 'Paralegal']) & ~lawyers_jobs_query_result['degree'].isin(['Unknown'])
]
datapoint_5 = only_senior_lawyers['degree'].value_counts().sort_values(ascending=False)
df5 = datapoint_5.reset_index().rename(columns={"index": 'degree', 0: 'count'})

plt.figure(figsize=(10, 5))
sns5 = sns.barplot(
    data=df5,
    y='degree',
    x='count',
    orient='h',
    edgecolor="black"
)

sns5.set(
    xlabel="Number of People",
    ylabel="Degree",
    title="Degrees of Lawyers (without Paralegals, Secretaries and Assistants)"
)

total5 = df5['count'].sum()
for element in sns5.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total5 * 100
    sns5.text(
        element.get_width() + total5*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df5['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout() 

### If we excluse these support and junior positions from the dataset, we, surprisingly, do not get a significant increase in the percentage of legal workers holding a law degree. On the contrary, senior legal workers (excluding administrative and support positions) are more likely to hold a business degree than a law degree.

# 7. Law Grads’ Work Fields

Question: Apart from law and IT, where do law grads work?W e’ll map their fields beyond tech.

In [ ]:
with open('/kaggle/input/sql-files/07_law_grads_work_fields.sql') as f:
    nine_query = f.read()

nine_query_result = pandasql.sqldf(nine_query, locals())
filtered_nine_query = nine_query_result[
    ~nine_query_result['field'].isin(['Unknown'])
]

datapoint_9 = filtered_nine_query['field'].value_counts().sort_values(ascending=False)
df9 = datapoint_9.reset_index().rename(columns={"index": 'field', 0: 'count'})

plt.figure(figsize=(10, 5))
sns9 = sns.barplot(
    data=df9,
    y='field',
    x='count',
    orient='h',
    edgecolor="black"
)

sns9.set(
    xlabel="Number of People",
    ylabel="Field",
    title="Fields of Work for Law Graduates"
)

total9 = df9['count'].sum()
for element in sns9.patches:
    count = int(element.get_width()) # width of the bar in the bar chart = count
    percentage = count/total9 * 100
    sns9.text(
        element.get_width() + total9*0.01, # x position
        element.get_y() + element.get_height()/2, # y position
        f"{count} ({percentage:.1f}%)",
        va="center" # vertical align
    )

plt.xlim(0, df9['count'].max() * 1.3) # xlim() sets the beginning and end of the x axis. This extends x-axis limit to give space for labels
plt.tight_layout()

# List: what is in "Other" in df9
filtered_nine_inv_query = nine_query_result[
    nine_query_result['field'].isin(['Other'])
]
datapoint_10 = filtered_nine_inv_query['title'].value_counts().sort_values(ascending=False)
df10 = datapoint_10.reset_index().rename(columns={"index": 'title', 0: 'count'})

# List: most common programs in df9
datapoint_11 = filtered_nine_query['program'].value_counts().sort_values(ascending=False)
df11 = datapoint_11.reset_index().rename(columns={"index": 'program', 0: 'count'})

# Run
plt.show()

### Interestingly, people holding law degrees are more likely to hold management or administrative positions, or work in IT, than work in the legal field.

# Summary Findings

As a result of the above study on career transitions, I believe that some of the most surprising and useful findings are:

- **Law is one of the least likely degree majors for an IT professional to have.**
- **In total, about 46% of all developers hold Computer Science and related degrees.**
- **Back-end developers are more likely to hold Computer Science and related degrees than Front-end developers and data scientists**, which suggests that Front-end and Data Science are more popular career transition targets than back-end development.
- **Surprisingly, the legal field has a smaller percentage of people holding law degrees.** This is true even when you take paralegals, secretaries, and assistants out of consideration.
